# Tapdance — Colab Web UI

在 Colab 上以纯 Web 模式运行 Tapdance。

**运行前准备（可选）：**
在左侧边栏 → 🔑 Secrets 中添加以下 secret（也可以跑起来后在应用内的「API 配置」页面填写）：
- `GEMINI_API_KEY` — Google AI Studio key
- `VOLCENGINE_API_KEY` — 火山引擎 Ark API key

> **注意：** 本地 Seedance CLI（Dreamina 二进制文件）无法在 Colab 上运行。视频生成请在应用内选择「火山引擎 Ark」执行器。

## Cell 1 — 安装 Node.js 22

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - 2>/dev/null
!sudo apt-get install -y nodejs 2>&1 | tail -3
!node --version && npm --version

## Cell 2 — 克隆仓库 & 安装依赖

`--ignore-scripts` 跳过 Electron 二进制下载（Web 模式不需要），加快安装速度。

In [ ]:
!git clone https://github.com/lazygunner/Tapdance.git /content/Tapdance
%cd /content/Tapdance
!npm install --ignore-scripts 2>&1 | tail -10
print("依赖安装完成")

## Cell 3 — 写 .env 文件（注入 API Key）

In [ ]:
from google.colab import userdata

def get_secret(key):
    try:
        return userdata.get(key) or ''
    except Exception:
        return ''

# 读取 key（secret 不存在时留空，稍后可在应用内填写）
GEMINI_API_KEY     = get_secret('GEMINI_API_KEY')
VOLCENGINE_API_KEY = get_secret('VOLCENGINE_API_KEY')

env_content = f"""GEMINI_API_KEY="{GEMINI_API_KEY}"
DISABLE_HMR=true
"""

with open('/content/Tapdance/.env', 'w') as f:
    f.write(env_content)

print("GEMINI_API_KEY:",     "已设置" if GEMINI_API_KEY     else "未设置（可在应用内填写）")
print("VOLCENGINE_API_KEY:", "已设置" if VOLCENGINE_API_KEY else "未设置（可在应用内填写）")

## Cell 4 — 启动 Vite Dev Server（后台）

In [ ]:
import subprocess, time

dev_proc = subprocess.Popen(
    ['npm', 'run', 'dev:web'],
    cwd='/content/Tapdance',
    stdout=open('/tmp/vite.log', 'w'),
    stderr=subprocess.STDOUT,
)

print("等待 Vite 启动...")
time.sleep(12)

with open('/tmp/vite.log') as f:
    log = f.read()

print(log)

if '3001' in log:
    print("✅ Vite dev server 已在 port 3001 启动")
else:
    print("⚠️  未检测到端口，请查看上方日志")

## Cell 5 — 开启公网隧道

运行后会打印一个 `https://xxxx.loca.lt` 格式的 URL，在浏览器中打开即可访问 Tapdance。

> 首次访问 loca.lt 链接时，会出现一个「点击继续」的确认页面，点击后正常进入应用。
>
> **本 Cell 会持续运行以保持隧道畅通，不要中断它。** 若需重新获取 URL，重跑本 Cell 即可。

In [ ]:
!npx --yes localtunnel --port 3001

---

## 进入应用后的配置步骤

1. 点击右上角 **「API 配置」**
2. 填写 **Gemini API Key**（用于文本生成和图像生成）
3. 填写 **火山引擎 Ark API Key** 和 **接入点 ID**（用于 SeedDance 2.0 视频生成）
   - API Key：你的 UUID 格式 key
   - 接入点 ID：`ep-20260416124751-x4tfn`
4. 视频执行器选择 **「Ark API」**（不要选 CLI，CLI 需要本地二进制文件）
5. 保存后即可开始使用

## 功能限制（Colab Web 模式）

| 功能 | 状态 |
|------|------|
| 创意视频流程（Brief → 分镜 → 视频） | ✅ 可用 |
| 极速成片流程 | ✅ 可用 |
| 广告视频流程 | ✅ 可用 |
| 火山引擎 Ark 视频生成 | ✅ 可用 |
| Gemini 文本 / 图像生成 | ✅ 可用 |
| 本地 Seedance CLI（Dreamina） | ❌ 不可用 |
| 本地文件系统持久化（SQLite） | ❌ 不可用 |